# IT Service Ticket Classification — First Fine-Tuning

## Introduction

This notebook performs the first fine-tuning experiment for a multi-class IT service ticket classification model using `distilbert-base-uncased`.

The goal of this stage is to train a functional baseline model using the tokenized Hugging Face dataset created in Day 2. This notebook uses the Hugging Face `Trainer` API, evaluates the model with accuracy and macro F1-score, and saves the trained model artifact for later testing and deployment.

This is not intended to be the final optimized model. Instead, it validates the complete training workflow and creates a first real model that can be evaluated and improved in the next stage.

## 1. Install Required Libraries

In [ ]:
# Install the Hugging Face Datasets library to load the tokenized dataset saved in Day 2.
!pip install datasets

# Install the Hugging Face Transformers library to load DistilBERT and use the Trainer API.
!pip install transformers

# Install the Evaluate library to calculate accuracy and F1-score.
!pip install evaluate

# Install scikit-learn because some evaluation metrics depend on it internally.
!pip install scikit-learn

# Install PyTorch, which is required to train transformer models.
!pip install torch

## 2. Import Libraries

In [ ]:
# Import os to work with folders and file paths.
import os

# Import json to load the label mappings created in Day 2.
import json

# Import numpy to convert model logits into predicted class IDs.
import numpy as np

# Import torch to check whether GPU acceleration is available.
import torch

# Import load_from_disk to load the tokenized Hugging Face DatasetDict saved in Day 2.
from datasets import load_from_disk

# Import evaluate to calculate model evaluation metrics.
import evaluate

# Import AutoTokenizer to reload the same tokenizer used during preprocessing.
from transformers import AutoTokenizer

# Import AutoModelForSequenceClassification to load DistilBERT with a classification head.
from transformers import AutoModelForSequenceClassification

# Import TrainingArguments to define the training configuration.
from transformers import TrainingArguments

# Import Trainer to manage the fine-tuning workflow.
from transformers import Trainer

# Import DataCollatorWithPadding to dynamically pad batches during training.
from transformers import DataCollatorWithPadding

## 3. Define Project Paths

In [ ]:
# Define the path where the tokenized Hugging Face DatasetDict was saved in Day 2.
dataset_path = "../data/tokenized/tickets_distilbert"

# Define the path where the label-to-ID mapping was saved in Day 2.
label2id_path = "../data/metadata/label2id.json"

# Define the path where the ID-to-label mapping was saved in Day 2.
id2label_path = "../data/metadata/id2label.json"

# Define the folder where the fine-tuned model will be saved.
model_output_path = "../models/distilbert_ticket_classifier_v1"

## 4. Load Tokenized Dataset and Label Mappings

In [ ]:
# Load the tokenized Hugging Face DatasetDict from disk.
dataset = load_from_disk(dataset_path)

# Display the dataset structure to confirm that train, validation, and test splits are available.
dataset

In [ ]:
# Open the label-to-ID JSON file created in Day 2.
with open(label2id_path, "r") as file:
    # Load the label-to-ID mapping into a Python dictionary.
    label2id = json.load(file)

# Open the ID-to-label JSON file created in Day 2.
with open(id2label_path, "r") as file:
    # Load the ID-to-label mapping into a Python dictionary.
    id2label = json.load(file)

# Convert JSON keys from strings to integers because model configuration expects integer label IDs.
id2label = {int(key): value for key, value in id2label.items()}

# Count the number of target classes in the classification problem.
num_labels = len(label2id)

# Display the number of labels to verify the classification setup.
num_labels

In [ ]:
# Display the number of examples in the training split.
print("Train examples:", len(dataset["train"]))

# Display the number of examples in the validation split.
print("Validation examples:", len(dataset["validation"]))

# Display the number of examples in the test split.
print("Test examples:", len(dataset["test"]))

## 5. Load Tokenizer and Model

In [ ]:
# Define the same pretrained checkpoint used during Day 2 tokenization.
model_checkpoint = "distilbert-base-uncased"

# Load the tokenizer associated with the selected checkpoint.
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Load DistilBERT with a sequence classification head for the number of ticket categories.
model = AutoModelForSequenceClassification.from_pretrained(
    # Use the pretrained DistilBERT checkpoint as the base model.
    model_checkpoint,
    # Set the number of output classes for multi-class classification.
    num_labels=num_labels,
    # Provide the ID-to-label mapping for readable model outputs.
    id2label=id2label,
    # Provide the label-to-ID mapping for consistent training configuration.
    label2id=label2id
)

In [ ]:
# Check whether a GPU is available for faster training.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Display the device that will be used by the training process.
print("Training device:", device)

# Move the model to the selected device.
model.to(device)

## 6. Define Evaluation Metrics

In [ ]:
# Load the accuracy metric from the Evaluate library.
accuracy_metric = evaluate.load("accuracy")

# Load the F1-score metric from the Evaluate library.
f1_metric = evaluate.load("f1")

# Define a function that calculates evaluation metrics during validation.
def compute_metrics(eval_pred):
    # Unpack the model outputs and true labels from the evaluation prediction object.
    logits, labels = eval_pred

    # Convert logits into predicted class IDs by selecting the class with the highest score.
    predictions = np.argmax(logits, axis=-1)

    # Calculate accuracy using the predicted labels and true labels.
    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )["accuracy"]

    # Calculate macro F1-score to evaluate performance across all classes equally.
    f1_macro = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )["f1"]

    # Return the metrics as a dictionary for the Trainer API.
    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro
    }

## 7. Configure Training Arguments

In [ ]:
# Define the training configuration for the first fine-tuning experiment.
training_args = TrainingArguments(
    # Save checkpoints and training outputs in the model output folder.
    output_dir=model_output_path,
    # Set the initial learning rate for fine-tuning.
    learning_rate=2e-5,
    # Set the batch size used by each device during training.
    per_device_train_batch_size=16,
    # Set the batch size used by each device during evaluation.
    per_device_eval_batch_size=16,
    # Train the model for two epochs as the first baseline experiment.
    num_train_epochs=2,
    # Evaluate the model at the end of each epoch.
    eval_strategy="epoch",
    # Save a checkpoint at the end of each epoch.
    save_strategy="epoch",
    # Log training progress at the end of each epoch.
    logging_strategy="epoch",
    # Load the best checkpoint at the end of training.
    load_best_model_at_end=True,
    # Use macro F1-score to select the best model checkpoint.
    metric_for_best_model="f1_macro",
    # Higher macro F1-score indicates better model performance.
    greater_is_better=True,
    # Disable external experiment tracking integrations for this first version.
    report_to="none"
)

## 8. Create Data Collator and Trainer

In [ ]:
# Create a data collator that dynamically pads each batch during training.
data_collator = DataCollatorWithPadding(
    # Use the same tokenizer that was used for preprocessing.
    tokenizer=tokenizer
)

# Initialize the Hugging Face Trainer.
trainer = Trainer(
    # Pass the model that will be fine-tuned.
    model=model,
    # Pass the training configuration.
    args=training_args,
    # Provide the tokenized training dataset.
    train_dataset=dataset["train"],
    # Provide the tokenized validation dataset.
    eval_dataset=dataset["validation"],
    # Provide the data collator for batch preparation.
    data_collator=data_collator,
    # Provide the metric function for validation evaluation.
    compute_metrics=compute_metrics
)

## 9. Train the Model

In [ ]:
# Start the fine-tuning process using the training dataset and validation dataset.
trainer.train()

## 10. Evaluate the First Model

In [ ]:
# Evaluate the trained model on the validation split.
validation_results = trainer.evaluate(eval_dataset=dataset["validation"])

# Display validation results.
validation_results

In [ ]:
# Evaluate the trained model on the test split for a first generalization check.
test_results = trainer.evaluate(eval_dataset=dataset["test"])

# Display test results.
test_results

## 11. Save the Fine-Tuned Model

In [ ]:
# Create the model output folder if it does not already exist.
os.makedirs(model_output_path, exist_ok=True)

# Save the fine-tuned model to disk.
trainer.save_model(model_output_path)

# Save the tokenizer with the model to make future inference easier.
tokenizer.save_pretrained(model_output_path)

# Confirm where the model was saved.
print("Model saved to:", model_output_path)

## 12. First Fine-Tuning Conclusions

In this stage, a first functional version of the IT service ticket classification model was trained using `distilbert-base-uncased`.

The objective was not to fully optimize the model yet, but to validate the full fine-tuning workflow: loading the processed Hugging Face dataset, configuring the Trainer API, training the model, evaluating the first results, and saving the trained artifact.

The model was evaluated using accuracy and macro F1-score. Macro F1-score is especially useful in this project because the task is multi-class classification and each class should contribute equally to the evaluation, even when class frequencies are not perfectly balanced.

This first model will be used as a baseline for the next stage, where class-level performance, confusion matrix results, and common prediction errors will be analyzed.